In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:
import pandas as pd
import numpy as np

# Example: create dummy predictions
# Replace `test.csv` with your actual test file if needed

sample = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv")
sample.to_csv("submission.csv",index=False)

In [3]:
df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 8 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   id      2000 non-null   int64 
 1   prompt  2000 non-null   object
 2   A       2000 non-null   object
 3   B       2000 non-null   object
 4   C       2000 non-null   object
 5   D       2000 non-null   object
 6   E       2000 non-null   object
 7   answer  2000 non-null   object
dtypes: int64(1), object(7)
memory usage: 125.1+ KB


## Q1.
Calculate the frequency distribution of the correct  answer  (A, B, C, D, E) in train.csv. Based on your counts, what is the sum of the occurrences of the most frequent option and the least frequent option? 

In [8]:
result = df['answer'].value_counts().max() + df['answer'].value_counts().min()
print(result)

814


## q2.
After converting the prompt column to lowercase and removing all standard punctuation characters (using Python's string.punctuation), split the text by whitespace. What is the total number of unique words (vocabulary size) across the entire cleaned prompt column of train.csv? 

In [35]:
import string

translator = str.maketrans('', '', string.punctuation)

cleaned_prompts = (
    df['prompt']
    .str.lower()
    .str.translate(translator)
)
print(cleaned_prompts)

0       pick the best possible answer what is martin h...
1                what is acceleratorbased lightion fusion
2       determine the correct option what is the term ...
3       select the most accurate option what is martin...
4       identify the correct statement what is the con...
                              ...                        
1995    what is the piezoelectric strain coefficient f...
1996    identify the correct statement what is the syn...
1997    determine the correct option what does earnsha...
1998    identify the correct statement what is the rev...
1999    pick the best possible answer which of the fol...
Name: prompt, Length: 2000, dtype: object


In [36]:
vocab = set()

for text in cleaned_prompts:
    words = text.split()   # split by whitespace
    vocab.update(words)
vocab_size = len(vocab)
print(vocab_size)

859


## q3.
Using the cleaned prompt from Row ID 1, filter out the standard English stop words using sklearn.feature_extraction.text.ENGLISH_STOP_WORDS. How many words are left in the prompt for Row ID 1 after filtering? 

In [18]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
prompt = df.loc[df['id'] == 1, 'prompt'].iloc[0]
print(prompt)

Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the listed options.


In [23]:
translator = str.maketrans('', '', string.punctuation)

cleaned_prompt = prompt.lower().translate(translator)
print(cleaned_prompt)


words = cleaned_prompt.split()
print(words)

filtered_words = [word for word in words if word not in ENGLISH_STOP_WORDS]

count = len(filtered_words)

print(filtered_words)
print("Words remaining:", count)

pick the best possible answer what is martin heideggers view on the relationship between time and human existence among the listed options
['pick', 'the', 'best', 'possible', 'answer', 'what', 'is', 'martin', 'heideggers', 'view', 'on', 'the', 'relationship', 'between', 'time', 'and', 'human', 'existence', 'among', 'the', 'listed', 'options']
['pick', 'best', 'possible', 'answer', 'martin', 'heideggers', 'view', 'relationship', 'time', 'human', 'existence', 'listed', 'options']
Words remaining: 13


## q4.
Fit a default TfidfVectorizer(stop_words='english') on a list containing all the combined text of the prompts and options in train.csv. What is the exact total number of feature columns (vocabulary size) generated by the vectorizer?

In [24]:
from sklearn.feature_extraction.text import TfidfVectorizer

combined_text = (
    df['prompt'].fillna('') + ' ' +
    df['A'].fillna('') + ' ' +
    df['B'].fillna('') + ' ' +
    df['C'].fillna('') + ' ' +
    df['D'].fillna('') + ' ' +
    df['E'].fillna('')
)

vectorizer = TfidfVectorizer(stop_words='english')

X = vectorizer.fit_transform(combined_text)

print(X.shape)

(2000, 2762)


## q5.
Using the TF-IDF vectorizer fitted in Question 3, calculate the cosine similarity between the prompt and option A strictly for Row ID 1. What is the resulting similarity score? (Round to 4 decimal places).  

In [27]:
from sklearn.metrics.pairwise import cosine_similarity


row = df[df['id'] == 1].iloc[0]

prompt = row['prompt']
option_a = row['A']

prompt_vec = vectorizer.transform([prompt])
option_a_vec = vectorizer.transform([option_a])

similarity = cosine_similarity(prompt_vec, option_a_vec)[0][0]

print(similarity)
print(round(similarity, 4))

0.27202429519891635
0.272


## q6.
Expand the logic from Question 4: For every row in train.csv, calculate the cosine similarity between the prompt and each of its 5 options .  Then calculate the percentage of instances where the option with the highest cosine similarity matches the correct answer. 

In [28]:
correct = 0
total = len(df)

for _, row in df.iterrows():

    prompt_vec = vectorizer.transform([row['prompt']])

    similarities = {}

    for option in ['A', 'B', 'C', 'D', 'E']:
        option_vec = vectorizer.transform([row[option]])

        sim = cosine_similarity(prompt_vec, option_vec)[0][0]

        similarities[option] = sim

    predicted_answer = max(similarities, key=similarities.get)

    if predicted_answer == row['answer']:
        correct += 1


accuracy_percentage = (correct / total) * 100

print(f"{accuracy_percentage:.2f}%")

13.55%


## q7.
If the ground truth answer for a question is C, what is the MAP@3 score if a model predicts C A B?  

answer = 1

## q8.
If the ground truth answer for a question is  B, what is the MAP@3 score if a model predicts D B E?

answer = 0.5

## q9.
The Majority Class Baseline: Find the most frequent correct answer in the training set (using your data from Q1). Make a static prediction for every single row where that most frequent answer is your 1st guess, followed by the second most frequent, and then the third most frequent. What is the overall MAP@3 score of this "Majority Class" baseline on train.csv?

In [31]:
counts = df['answer'].value_counts()
print(counts)

top3 = counts.index[:3].tolist()

print(top3)

total_score = 0

for actual in df['answer']:

    if actual == top3[0]:
        total_score += 1

    elif actual == top3[1]:
        total_score += 1/2

    elif actual == top3[2]:
        total_score += 1/3

    else:
        total_score += 0

map3 = total_score / len(df)

print(f"{map3:.6f}")

answer
B    490
C    459
A    369
D    358
E    324
Name: count, dtype: int64
['B', 'C', 'A']
0.421250


## q10.
The TF-IDF Pipeline: Build a basic pipeline that evaluates every row in train.csv. For each row, calculate the TF-IDF cosine similarity between the prompt and each of the 5 options. Sort these options from highest similarity to lowest to form your top 3 predictions. What is the final average MAP@3 score of this TF-IDF pipeline across the entire training set?  

In [32]:
def ap_at_3(actual, predicted):
    """
    actual: correct answer ('A','B','C','D','E')
    predicted: list of top 3 predictions
    """
    
    for i, pred in enumerate(predicted[:3], start=1):
        if pred == actual:
            return 1 / i

    return 0.0

In [33]:
scores = []

for _, row in df.iterrows():

    texts = [
        row['prompt'],
        row['A'],
        row['B'],
        row['C'],
        row['D'],
        row['E']
    ]

    vectors = vectorizer.transform(texts)

    prompt_vec = vectors[0]
    option_vecs = vectors[1:]

    sims = cosine_similarity(prompt_vec, option_vecs)[0]

    options = ['A', 'B', 'C', 'D', 'E']

    ranked_options = [
        option
        for _, option in sorted(
            zip(sims, options),
            reverse=True
        )
    ]

    top3 = ranked_options[:3]

    score = ap_at_3(row['answer'], top3)

    scores.append(score)

In [34]:
map3 = sum(scores) / len(scores)

print(f"MAP@3 = {map3:.6f}")

MAP@3 = 0.256417
